# CAS Exam 5: Expected Claims Method (Using Friedland Industry Auto Data)

This notebook uses:
- `C:/Users/cphel/Documents/code/exam_5/chainladder-python/chainladder/utils/data/friedland_us_industry_auto.csv`

Learning goals:
- Show expected claims method concepts with the same dataset used in the chain ladder notebook.
- Use `chainladder.Triangle` for the claims side, then apply ECR/pure-premium priors for ultimate and IBNR.
- Compare provided ECR vs derived ECR approaches with actuarial considerations.


## Formula-Sheet Core Equations

Premium basis:
- Ultimate Claims = (Expected Claims / Earned Premium) x Earned Premium
- ECR = Expected Claims / Earned Premium

Exposure basis:
- Ultimate Claims = (Expected Claims / Earned Exposure) x Earned Exposure
- Expected Pure Premium = Expected Claims / Earned Exposure

Interpretation:
- Expected claims method uses a priori assumptions (ECR/pure premium) more heavily than immature emergence patterns.


---
## Formula-Sheet Reference: Expected Claims Method

### Process Overview (Friedland Ch. 5)

| Step | Action | Key Actuarial Decision |
|------|--------|------------------------|
| 1 | Obtain earned premium (or exposure) by AY | Must be on a consistent rate/coverage basis |
| 2 | Select ECR (or Pure Premium) | Provided a priori, or derived from historical experience |
| 3 | Apply level adjustments if needed | Rate changes, operational changes, coverage shifts |
| 4 | Compute expected ultimate: `Ultimate = ECR × EP` | Or `PP × Exposure` on exposure basis |
| 5 | Calculate IBNR: `IBNR = Ultimate − Paid (or Reported)` | Paid basis gives larger IBNR; reported basis is more common |

### Key Formulas

**Premium basis:**
$$\text{Expected Ultimate}_i = \text{ECR} \times \text{Earned Premium}_i$$
$$\text{ECR} = \frac{\text{Expected Claims}}{\text{Earned Premium}}$$

**Exposure basis:**
$$\text{Expected Ultimate}_i = \overline{PP} \times \text{Earned Exposure}_i$$
$$\overline{PP} = \frac{\text{Expected Claims}}{\text{Earned Exposure}}$$

**IBNR (paid basis):**
$$\text{IBNR}_i = \text{Expected Ultimate}_i - \text{Paid to Date}_i$$

**IBNR (reported basis):**
$$\text{IBNR}_i = \text{Expected Ultimate}_i - \text{Reported to Date}_i$$

### ECR Derivation Approaches

| Method | Formula | When to Use |
|--------|---------|-------------|
| Arithmetic average | $\bar{r} = \frac{1}{n}\sum r_i$ | Equal weight per AY; best when premium volumes are similar across years |
| Median | $\text{med}(r_1, \ldots, r_n)$ | Robust to a single outlier year |
| Volume-weighted | $\frac{\sum \text{Claims}_i}{\sum \text{Premium}_i}$ | **Exam default — preferred when premium volumes differ significantly across AYs** |

### Level Adjustment

If the historical ECR was calibrated to an older rate or coverage level:
$$\text{Adjusted ECR} = \text{Historical ECR} \times \text{Level Adjustment Factor}$$

### Connection to BF and Chain Ladder

The three methods form a credibility spectrum:

$$\underbrace{\text{Expected Claims}}_{\text{100\% prior weight}} \longrightarrow \underbrace{\text{BF}}_{\text{blend}} \longrightarrow \underbrace{\text{Chain Ladder}}_{\text{100\% emergence weight}}$$

**BF formula:**
$$\text{BF Ultimate}_i = \text{Reported}_i + \underbrace{(1 - \hat{q}_i)}_{\text{\% unreported}} \times \underbrace{\text{Expected Ultimate}_i}_{\text{a priori}}$$

- When $\hat{q}_i \approx 0$ (very immature): BF $\approx$ Expected Ultimate — prior dominates
- When $\hat{q}_i \approx 1$ (very mature): BF $\approx$ Reported — chain ladder dominates

> **Exam insight:** The expected claims method is the limiting case of BF where 100% weight is placed on the a priori. BF is almost always preferred over pure expected claims unless the line is brand new (no development history whatsoever).

### When to Use Expected Claims (vs Chain Ladder vs BF)

| Situation | Preferred Method | Reason |
|-----------|-----------------|--------|
| Very immature AY (≤ 24 months) | **Expected Claims** | CDF enormous; leverage risk dominates actual data |
| New line or new territory | **Expected Claims** | No credible development history exists |
| Major operational change | **Expected Claims** | Historical LDFs no longer applicable |
| Moderate maturity (24–60 months) | **BF** | Growing credibility of actual emergence; blend is optimal |
| Mature, stable AY | **Chain Ladder** | Actual development is fully credible |

### Key Assumptions
1. Selected ECR reliably represents the long-run loss ratio for each projection AY
2. Earned premium is on a consistent rate and coverage basis
3. Level adjustments correctly account for rate changes between calibration and projection years
4. The method assigns **equal weight** to each AY regardless of maturity — no credibility weighting by development age

In [17]:
from __future__ import annotations

from pathlib import Path
import sys

import chainladder as cl
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from reservingengine.reserving import (
    adjust_selected_ecr,
    build_environment_impact_table,
    expected_claims_from_exposure,
    expected_claims_from_premium,
    ibnr_from_expected_claims,
    implied_ecr,
    selected_ecr_from_history,
)

DATA_PATH = ROOT / 'chainladder-python' / 'chainladder' / 'utils' / 'data' / 'friedland_us_industry_auto.csv'
raw = pd.read_csv(DATA_PATH)

triangle = cl.Triangle(
    raw,
    origin='Accident Year',
    development='Calendar Year',
    columns=['Paid Claims', 'Reported Claims'],
    cumulative=True,
)
paid_triangle = triangle['Paid Claims']
reported_triangle = triangle['Reported Claims']

raw.head(), {'triangle_shape': triangle.shape, 'valuation_date': str(triangle.valuation_date)}


(   Accident Year  Calendar Year  Paid Claims  Reported Claims
 0           1998           1998     18539254         37017487
 1           1998           1999     33231039         43169009
 2           1998           2000     40062008         45568919
 3           1998           2001     43892039         46784558
 4           1998           2002     45896535         47337318,
 {'triangle_shape': (1, 2, 10, 10),
  'valuation_date': '2007-12-31 23:59:59.999999999'})

## Step 1: Load Data and Understand Development Status

The Friedland industry auto dataset provides paid and reported claims for accident years 1998–2007 across 10 development ages (12–120 months). Unlike the chain ladder method, expected claims **does not project development factors** — it replaces that process entirely with a selected ECR.

However, understanding how developed each AY is serves two purposes:
1. **Identify credible calibration years** — only mature AYs (near their ultimate) should anchor the historical ECR derivation
2. **Understand IBNR composition** — immature AYs will have the largest IBNR as a share of expected ultimate, and are where the expected claims method is most important

The table below shows each AY's latest diagonal position at valuation date 12/31/2007.

In [18]:
paid_long = paid_triangle.to_frame(origin_as_datetime=False, keepdims=True).reset_index()
reported_long = reported_triangle.to_frame(origin_as_datetime=False, keepdims=True).reset_index()
paid_matrix = paid_long.pivot(index='origin', columns='development', values='Paid Claims').sort_index().sort_index(axis=1)
reported_matrix = reported_long.pivot(index='origin', columns='development', values='Reported Claims').sort_index().sort_index(axis=1)

latest_paid = paid_triangle.latest_diagonal.to_frame().iloc[:, 0]
latest_reported = reported_triangle.latest_diagonal.to_frame().iloc[:, 0]
latest_age = paid_matrix.notna().iloc[:, ::-1].idxmax(axis=1)

claim_inputs = pd.DataFrame(
    {
        'LatestPaid': latest_paid.values,
        'LatestReported': latest_reported.values,
        'LatestDevelopmentAge': latest_age.values,
    },
    index=latest_paid.index.year,
)
claim_inputs.index.name = 'AccidentYear'
claim_inputs


,LatestPaid,LatestReported,LatestDevelopmentAge
AccidentYear,,,
1998,47644187.0,47742304.0,120
1999,51000534.0,51185767.0,108
2000,54533225.0,54837929.0,96
2001,55878421.0,56299562.0,84
2002,57807215.0,58592712.0,72
2003,55930654.0,57565344.0,60
2004,53774672.0,56976657.0,48
2005,50644994.0,56786410.0,36
2006,43606497.0,54641339.0,24


### Development Status: % Paid of Ultimate

For context, a quick volume-weighted chain ladder is run to estimate ultimate and compute `% Paid` and `% Unreported` for each AY. This is **not** the expected claims method — it is purely diagnostic, illustrating why the expected claims approach is appropriate for the most recent years.

> **Exam connection:** High `% Unreported` = high chain ladder leverage = expected claims (or BF) preferred over pure development. AY 2007 at 12 months has the largest unreported fraction; applying a chain ladder CDF there would amplify any LDF selection error dramatically.

In [ ]:
# Quick chain ladder ultimate estimate — used only to compute % paid / % unreported for context
# This shows WHY expected claims is needed for immature AYs: very high unreported fraction
_dev_fitted = cl.Development(average='volume').fit_transform(paid_triangle)
_cl_ult_frame = cl.Chainladder().fit(_dev_fitted).ultimate_.to_frame()
_cl_ult = _cl_ult_frame.iloc[:, 0]
_cl_ult.index = [int(str(x)[:4]) for x in _cl_ult.index]  # normalize to integer years

development_status = pd.DataFrame({
    'DevAge': claim_inputs['LatestDevelopmentAge'],
    'Paid': claim_inputs['LatestPaid'],
    'CL_Ultimate_Est': _cl_ult,
    'PctPaid': claim_inputs['LatestPaid'] / _cl_ult,
    'PctUnreported': 1 - claim_inputs['LatestPaid'] / _cl_ult,
    'PaidToReported': claim_inputs['LatestPaid'] / claim_inputs['LatestReported'],
})

development_status.style.format({
    'Paid': '{:,.0f}',
    'CL_Ultimate_Est': '{:,.0f}',
    'PctPaid': '{:.1%}',
    'PctUnreported': '{:.1%}',
    'PaidToReported': '{:.1%}',
}).background_gradient(subset=['PctUnreported'], cmap='RdYlGn_r')

## Build Priors (Earned Premium / Exposure) Aligned to This Dataset

This CSV has paid and reported claims but not premium/exposure. For expected-claims demonstration, we add explicit prior assumptions by AY:
- pricing ECR assumption to infer earned premium scale,
- average premium per exposure to infer earned exposure.

These priors are the central modeling choice in the expected claims method.


In [19]:
ay = claim_inputs.index

pricing_ecr_assumption = pd.Series(
    [0.74, 0.74, 0.745, 0.75, 0.755, 0.76, 0.765, 0.77, 0.775, 0.78],
    index=ay,
    dtype=float,
)
earned_premium = claim_inputs['LatestReported'] / pricing_ecr_assumption

avg_premium_per_exposure = pd.Series(
    [4700, 4750, 4800, 4850, 4900, 4950, 5000, 5050, 5100, 5150],
    index=ay,
    dtype=float,
)
earned_exposure = earned_premium / avg_premium_per_exposure

prior_inputs = pd.DataFrame(
    {
        'PricingECRAssumption': pricing_ecr_assumption,
        'EarnedPremium': earned_premium,
        'AvgPremiumPerExposure': avg_premium_per_exposure,
        'EarnedExposure': earned_exposure,
    }
)
prior_inputs


,PricingECRAssumption,EarnedPremium,AvgPremiumPerExposure,EarnedExposure
AccidentYear,,,,
1998,0.740,6.451663e+07,4700.0,13726.941921
1999,0.740,6.916996e+07,4750.0,14562.095875
2000,0.745,7.360796e+07,4800.0,15334.991331
2001,0.750,7.506608e+07,4850.0,15477.542818
2002,0.755,7.760624e+07,4900.0,15838.008380
2003,0.760,7.574387e+07,4950.0,15301.792663
2004,0.765,7.447929e+07,5000.0,14895.858039
2005,0.770,7.374858e+07,5050.0,14603.680082
2006,0.775,7.050495e+07,5100.0,13824.500696


## Provided ECR Case and Exposure-Basis Cross-Check

Exam guideline reminder:
- If provided ECR is not tied to a specific AY, treat it as broadly applicable unless stated otherwise.

We project expected ultimate with a provided ECR, then show the equivalent exposure-basis framing.


In [20]:
provided_ecr = 0.78
expected_ultimate_provided = expected_claims_from_premium(earned_premium, provided_ecr)

selected_pure_premium = float(expected_ultimate_provided.sum() / earned_exposure.sum())
expected_ultimate_exposure = expected_claims_from_exposure(earned_exposure, selected_pure_premium)

provided_view = pd.DataFrame(
    {
        'EarnedPremium': earned_premium,
        'ExpectedUltimate_ProvidedECR': expected_ultimate_provided,
        'ExpectedUltimate_ExposureBasis': expected_ultimate_exposure,
    }
)
provided_view.loc['Total'] = provided_view.sum()
provided_view


,EarnedPremium,ExpectedUltimate_ProvidedECR,ExpectedUltimate_ExposureBasis
AccidentYear,,,
1998,6.451663e+07,5.032297e+07,5.268578e+07
1999,6.916996e+07,5.395257e+07,5.589121e+07
2000,7.360796e+07,5.741421e+07,5.885768e+07
2001,7.506608e+07,5.855154e+07,5.940481e+07
2002,7.760624e+07,6.053287e+07,6.078833e+07
2003,7.574387e+07,5.908022e+07,5.873026e+07
2004,7.447929e+07,5.809385e+07,5.717223e+07
2005,7.374858e+07,5.752390e+07,5.605082e+07
2006,7.050495e+07,5.499386e+07,5.306022e+07


### Provided ECR: Actuarial Considerations

**Premium vs exposure basis:** Both approaches produce the same **aggregate** expected ultimate — they are mathematically equivalent at the total level. Differences appear at the AY level when the average premium per exposure shifts over time (as it does here, trending from \$4,700 to \$5,150).

**When is a flat provided ECR appropriate?**
- Regulatory rate filings that specify a target loss ratio
- Management-directed a priori assumption for a new or restructured line
- Reinsurance pricing where the cedant provides a blended industry loss ratio

**Sensitivity risk of a flat ECR:** Applying one ECR to all AYs ignores any trend in loss ratios over time. If the true ECR is drifting (social inflation, medical cost trends, mix changes), a flat ECR will systematically over- or under-state ultimates for early vs late AYs. This is why year-specific ECRs (as shown in the final comparison section) are often more defensible.

## Determine ECR from Historical Experience

Formula-sheet process:
1. Adjust historical data as needed.
2. Calculate claim ratios.
3. Select ECR (arithmetic, median, volume-weighted).

Here we use mature AYs (development age >= 84) and latest reported as a proxy for near-ultimate historical claims.


In [21]:
mature_mask = claim_inputs['LatestDevelopmentAge'] >= 84
historical_claims_proxy = claim_inputs.loc[mature_mask, 'LatestReported']
historical_premium = earned_premium.loc[mature_mask]

historical_ecr = implied_ecr(historical_claims_proxy, historical_premium)
selected_ecrs = pd.Series(
    {
        'ArithmeticECR': selected_ecr_from_history(historical_claims_proxy, historical_premium, method='arithmetic'),
        'MedianECR': selected_ecr_from_history(historical_claims_proxy, historical_premium, method='median'),
        'VolumeWeightedECR': selected_ecr_from_history(historical_claims_proxy, historical_premium, method='volume_weighted'),
    }
)

historical_review = pd.DataFrame(
    {
        'HistoricalClaimsProxy': historical_claims_proxy,
        'HistoricalPremium': historical_premium,
        'ImpliedHistoricalECR': historical_ecr,
    }
)
historical_review, selected_ecrs


(              HistoricalClaimsProxy  HistoricalPremium  ImpliedHistoricalECR
 AccidentYear                                                                
 1998                     47742304.0       6.451663e+07                 0.740
 1999                     51185767.0       6.916996e+07                 0.740
 2000                     54837929.0       7.360796e+07                 0.745
 2001                     56299562.0       7.506608e+07                 0.750,
 ArithmeticECR        0.743750
 MedianECR            0.742500
 VolumeWeightedECR    0.743962
 dtype: float64)

### Step 3: ECR Selection — Actuarial Judgment

The three selection methods above typically give similar results for stable lines. Key exam judgment points:

**Why volume-weighted is the exam default:**
- Gives implicit credibility proportional to earned premium size
- Higher-premium years (often more recent) receive more weight — appropriate if the line is growing or rates have changed
- Most robust to years with abnormal volume

**Calibration window judgment:**
- Only use **mature AYs** (development age ≥ 84 months for auto liability) — immature AYs have unreported claims that bias the implied ECR downward
- In this dataset, AYs 1998–2001 (84–120 months at valuation) are credible calibration years; AYs 2002+ are excluded because they are still developing
- A 4–5 year window is typical; longer windows introduce trend distortion from older rate levels

**When derived ECR differs significantly from provided ECR:**
- Investigate whether rate level changes between calibration years and projection years explain the gap
- Apply an on-level adjustment before comparing (this is the "level adjustment" step below)
- If both are reasonable, the difference quantifies ECR parameter uncertainty — a key driver of reserve range

## Apply ECR Level Adjustment and Project IBNR

General guideline from the sheet:
- If premium level differs between historical calibration and projection AY, adjust ECR level before applying.

Below uses volume-weighted historical ECR with a 3% level adjustment.


In [22]:
base_ecr = float(selected_ecrs['VolumeWeightedECR'])
level_adjustment = 1.03
adjusted_ecr = adjust_selected_ecr(base_ecr, level_adjustment=level_adjustment)

expected_ultimate_derived = expected_claims_from_premium(earned_premium, adjusted_ecr)
paid_to_date = claim_inputs['LatestPaid']
ibnr_derived = ibnr_from_expected_claims(expected_ultimate_derived, paid_to_date)

derived_projection = pd.DataFrame(
    {
        'PaidToDate': paid_to_date,
        'ExpectedUltimate_Derived': expected_ultimate_derived,
        'IBNR_Derived': ibnr_derived,
    }
)
derived_projection.loc['Total'] = derived_projection.sum()
pd.Series({'BaseECR': base_ecr, 'LevelAdjustment': level_adjustment, 'AdjustedECR': adjusted_ecr}), derived_projection


(BaseECR            0.743962
 LevelAdjustment    1.030000
 AdjustedECR        0.766281
 dtype: float64,
                PaidToDate  ExpectedUltimate_Derived  IBNR_Derived
 AccidentYear                                                     
 1998           47644187.0              4.943785e+07  1.793667e+06
 1999           51000534.0              5.300361e+07  2.003076e+06
 2000           54533225.0              5.640437e+07  1.871141e+06
 2001           55878421.0              5.752170e+07  1.643278e+06
 2002           57807215.0              5.946817e+07  1.660959e+06
 2003           55930654.0              5.804108e+07  2.110423e+06
 2004           53774672.0              5.707205e+07  3.297379e+06
 2005           50644994.0              5.651213e+07  5.867131e+06
 2006           43606497.0              5.402659e+07  1.042010e+07
 2007           27229969.0              4.799429e+07  2.076432e+07
 Total         498050368.0              5.494818e+08  5.143147e+07)

### Step 5: Extended Projection Table — IBNR Composition by AY

This table mirrors the chain ladder "extended projection" format. The `% Paid` and `% Unreported` columns show how much of the expected ultimate is still outstanding for each AY.

**Key observations:**
- `% Unreported` is high for immature AYs — most of the reserve is driven by the a priori ECR, not by actual development
- For mature AYs (small % unreported), the IBNR is small and corresponds mostly to case-outstanding run-off
- The `% Unreported` column equals the $(1 - \hat{q}_i)$ weight in the BF formula — this is how BF "blends" the two methods

**Paid vs reported basis:**  
Using paid-to-date gives larger IBNR because paid lags reported. Using reported-to-date is more common in practice (and in Friedland exam problems) because it credits case reserves already established.

In [ ]:
# Extended projection table: mirrors the chain ladder "extended" format
# Adds % Paid and % Unreported columns relative to Expected Ultimate
# These correspond directly to the (1 - q) term in the BF formula

exp_ult = expected_ultimate_derived.copy()
ibnr_reported_basis = ibnr_from_expected_claims(exp_ult, claim_inputs['LatestReported'])

extended = pd.DataFrame({
    'EarnedPremium': earned_premium,
    'ExpectedUltimate': exp_ult,
    'PaidToDate': claim_inputs['LatestPaid'],
    'ReportedToDate': claim_inputs['LatestReported'],
    'IBNR_PaidBasis': ibnr_derived,
    'IBNR_ReportedBasis': ibnr_reported_basis,
    'PctPaid': claim_inputs['LatestPaid'] / exp_ult,
    'PctUnreported': 1 - claim_inputs['LatestPaid'] / exp_ult,
})

totals = extended[['EarnedPremium', 'ExpectedUltimate', 'PaidToDate',
                    'ReportedToDate', 'IBNR_PaidBasis', 'IBNR_ReportedBasis']].sum()
extended.loc['Total'] = totals

(
    pd.Series({
        'Applied ECR (adjusted)': adjusted_ecr,
        'Total IBNR — Paid Basis ($M)': ibnr_derived.sum() / 1e6,
        'Total IBNR — Reported Basis ($M)': ibnr_reported_basis.sum() / 1e6,
    }).round(4),
    extended.style.format({
        'EarnedPremium': '{:,.0f}',
        'ExpectedUltimate': '{:,.0f}',
        'PaidToDate': '{:,.0f}',
        'ReportedToDate': '{:,.0f}',
        'IBNR_PaidBasis': '{:,.0f}',
        'IBNR_ReportedBasis': '{:,.0f}',
        'PctPaid': '{:.1%}',
        'PctUnreported': '{:.1%}',
    }).background_gradient(subset=['PctUnreported'], cmap='RdYlGn_r')
)

## Assumptions, Use Cases, and Environmental Impacts

Key assumptions:
1. A priori estimate is more reliable than immature emergence.
2. Current paid/reported to date may have limited predictive power for ultimate on young AYs.

Works well when:
- new line/territory has limited historical development credibility,
- operational changes reduce comparability of historical development,
- early maturity development factors are highly leveraged.


In [23]:
impact_table = build_environment_impact_table()
impact_table


,Description,Paid impact,Reported impact
0,Increase in exposure,No material effect if average accident date is...,No material effect if average accident date is...
1,Average accident date shifts forward,Underestimates ultimate (usually less than dev...,Underestimates ultimate (usually less than dev...
2,Increase claim ratios,"If not reflected in selected ECR, ultimates ar...","If not reflected in selected ECR, ultimates ar..."
3,Speedup in claim settlement rate,Overestimates ultimate (usually less than deve...,No material effect
4,Increase in case outstanding adequacy,No material effect,Overestimates ultimate (usually less than deve...
5,Change in product mix,Impacted when segments have different ECRs/dev...,Impacted when segments have different ECRs/dev...


---
## Bornhuetter-Ferguson Connection: Expected Claims as the A Priori

The expected claims method is the foundation of the BF method. Understanding this connection is one of the most important concepts for Exam 5.

### BF Formula

$$\text{BF Ultimate}_i = \text{Reported}_i + \underbrace{(1 - \hat{q}_i)}_{\text{\% unreported}} \times \underbrace{\text{Expected Ultimate}_i}_{\text{a priori from expected claims}}$$

where $\hat{q}_i$ = cumulative % paid (or % reported) at latest evaluation for AY $i$.

### Credibility Spectrum

| Method | Formula | Weight on A Priori | Weight on Actual Emergence |
|--------|---------|-------------------|----------------------------|
| Expected Claims | $\text{Exp. Ultimate}$ | **100%** | 0% |
| BF | $\text{Reported} + (1-q) \times \text{Exp. Ult}$ | $(1-q) \times 100\%$ | $q \times 100\%$ |
| Chain Ladder | $\text{Reported} \times \text{CDF}$ | 0% | **100%** |

> As an AY matures ($q \to 1$), BF naturally converges to chain ladder. For brand-new AYs ($q \approx 0$), BF produces almost the same answer as expected claims.

### Worked Example (AY 2007 at 12 months, % paid ≈ 57% of expected ultimate)

Using `adjusted_ecr ≈ 0.7663` and `earned_premium(2007) ≈ $62.6M`:

$$\text{Expected Ultimate}_{2007} = 0.7663 \times \$62.6M \approx \$48.0M$$
$$\text{BF Ultimate}_{2007} = \$48.9M_{\text{reported}} + (1 - 0.57) \times \$48.0M \approx \$69.5M$$

Compare:
- **Expected Claims alone:** $\$48.0M$ — 100% weight on prior, ignores that $\$48.9M$ is already reported
- **BF:** $\$69.5M$ — credits the $\$48.9M$ reported plus projects remaining unreported via the prior
- **Chain Ladder:** would apply a very large CDF to just $\$27.2M$ paid — highly leveraged

> **Exam insight:** For AY 2007, BF is clearly better than either extreme. Expected claims is used here only to supply the a priori input. Recognizing when to use which method — and articulating why — is a common exam question.

## Provided vs Derived ECR Comparison

Educational takeaway:
- Provided ECR approach is straightforward and transparent but sensitive to whether the supplied ratio reflects current conditions.
- Derived ECR approach is anchored in historical calibration but sensitive to calibration window, mix changes, and level adjustments.


In [24]:
year_specific_ecr = pd.Series(
    [0.77, 0.77, 0.775, 0.78, 0.785, 0.79, 0.795, 0.80, 0.805, 0.81],
    index=ay,
    dtype=float,
)

expected_ultimate_year_specific = expected_claims_from_premium(earned_premium, year_specific_ecr)
ibnr_provided = ibnr_from_expected_claims(expected_ultimate_provided, paid_to_date)
ibnr_year_specific = ibnr_from_expected_claims(expected_ultimate_year_specific, paid_to_date)

comparison = pd.DataFrame(
    {
        'Ultimate_ProvidedECR': expected_ultimate_provided,
        'Ultimate_DerivedAdjustedECR': expected_ultimate_derived,
        'Ultimate_YearSpecificECR': expected_ultimate_year_specific,
        'IBNR_ProvidedECR': ibnr_provided,
        'IBNR_DerivedAdjustedECR': ibnr_derived,
        'IBNR_YearSpecificECR': ibnr_year_specific,
    }
)
comparison.loc['Total'] = comparison.sum()
comparison


,Ultimate_ProvidedECR,Ultimate_DerivedAdjustedECR,Ultimate_YearSpecificECR,IBNR_ProvidedECR,IBNR_DerivedAdjustedECR,IBNR_YearSpecificECR
AccidentYear,,,,,,
1998,5.032297e+07,4.943785e+07,4.967780e+07,2.678782e+06,1.793667e+06,2.033616e+06
1999,5.395257e+07,5.300361e+07,5.326087e+07,2.952031e+06,2.003076e+06,2.260332e+06
2000,5.741421e+07,5.640437e+07,5.704617e+07,2.880983e+06,1.871141e+06,2.512943e+06
2001,5.855154e+07,5.752170e+07,5.855154e+07,2.673123e+06,1.643278e+06,2.673123e+06
2002,6.053287e+07,5.946817e+07,6.092090e+07,2.725653e+06,1.660959e+06,3.113684e+06
2003,5.908022e+07,5.804108e+07,5.983766e+07,3.149567e+06,2.110423e+06,3.907006e+06
2004,5.809385e+07,5.707205e+07,5.921104e+07,4.319174e+06,3.297379e+06,5.436364e+06
2005,5.752390e+07,5.651213e+07,5.899887e+07,6.878902e+06,5.867131e+06,8.353874e+06
2006,5.499386e+07,5.402659e+07,5.675649e+07,1.138737e+07,1.042010e+07,1.314999e+07


---
## ECR Sensitivity Analysis: Reserve Range Across ECR Assumptions

The expected claims method's primary sensitivity is the selected ECR. Unlike chain ladder — where sensitivity flows through LDF selection and propagates differently by AY maturity — expected claims has a **direct, linear, and uniform** relationship between ECR and ultimate:

$$\Delta \text{Ultimate}_i = \Delta \text{ECR} \times \text{Earned Premium}_i$$

A 1% increase in ECR increases every AY's ultimate by exactly 1%, regardless of development age. This makes the reserve range easy to quantify but also means there is no "self-correcting" signal from actual claims emergence.

The sweep below shows how total IBNR and the IBNR for the three most immature AYs (2005–2007) vary across a plausible ECR range. The two reference points (derived + adjusted ECR and the flat provided ECR) should fall within this range.

In [ ]:
import numpy as np

# Sweep ECR from 0.70 to 0.86 in steps of 0.01
# Shows total IBNR (paid basis) and IBNR for the three most immature AYs (2005-2007)
ecr_sweep = np.round(np.arange(0.70, 0.87, 0.01), 4)
sensitivity_rows = []
for ecr_val in ecr_sweep:
    ult = expected_claims_from_premium(earned_premium, ecr_val)
    ibnr_series = ibnr_from_expected_claims(ult, claim_inputs['LatestPaid'])
    sensitivity_rows.append({
        'ECR': ecr_val,
        'TotalIBNR_M': ibnr_series.sum() / 1e6,
        'IBNR_AY2005_2007_M': ibnr_series.iloc[-3:].sum() / 1e6,
    })

sensitivity_df = pd.DataFrame(sensitivity_rows).set_index('ECR')

# Mark the two reference ECR values
ref_ecrs = {f'Derived+Adj ({adjusted_ecr:.4f})': adjusted_ecr, f'Provided ({provided_ecr:.2f})': provided_ecr}
print("Reference ECR assumptions:")
for label, val in ref_ecrs.items():
    closest = sensitivity_df.index[abs(sensitivity_df.index - val).argmin()]
    row = sensitivity_df.loc[closest]
    print(f"  {label:35s}  Total IBNR: ${row['TotalIBNR_M']:.1f}M  |  AY 2005-2007 IBNR: ${row['IBNR_AY2005_2007_M']:.1f}M")

print(f"\nReserve range across ECR {ecr_sweep.min():.2f}–{ecr_sweep.max():.2f}:")
print(f"  Total IBNR:         ${sensitivity_df['TotalIBNR_M'].min():.1f}M – ${sensitivity_df['TotalIBNR_M'].max():.1f}M")
print(f"  AY 2005-2007 IBNR:  ${sensitivity_df['IBNR_AY2005_2007_M'].min():.1f}M – ${sensitivity_df['IBNR_AY2005_2007_M'].max():.1f}M")
print()

sensitivity_df.rename(columns={
    'TotalIBNR_M': 'Total IBNR ($M)',
    'IBNR_AY2005_2007_M': 'IBNR AY2005–2007 ($M)',
}).style.format('{:.1f}').background_gradient(cmap='RdYlGn_r')

---
## Expected Claims Method: Exam Summary

### Strengths
- **Stable for immature AYs** — avoids the leverage problem inherent in chain ladder for young accident years
- **Transparent** — IBNR is directly proportional to ECR; easy to audit and explain to management
- **Appropriate for new lines** — when no development history exists, it is often the only credible option
- **Immune to development volatility** — case reserve changes, settlement rate shifts, and diagonal effects do not affect the ultimate estimate

### Weaknesses
- **100% weight on prior** — ignores actual claims emergence entirely, even when that data is informative
- **ECR sensitivity** — a 1% change in ECR produces a 1% change in every AY's ultimate with no differentiation by maturity
- **Level adjustment complexity** — requires careful on-leveling if historical ECR was calibrated to different rate levels
- **No self-correcting mechanism** — if the a priori was wrong, the method gives no signal; the error persists until the AY matures

### Exam Red Flags (when NOT to use expected claims)
- AY is well-developed (≥ 84 months for typical casualty lines) — chain ladder would use actual data more credibly
- You have reliable, stable development history with no operational changes
- The selected ECR is known to be stale or is inconsistent with current pricing levels

### Method Hierarchy for Exam 5

```
Less Mature ← AY Development Age → More Mature
Expected Claims  →  BF  →  Chain Ladder
(100% prior)      (blend)  (100% emergence)
```

| AY Maturity | Preferred Method | Rationale |
|-------------|-----------------|-----------|
| ≤ 24 months | Expected Claims | Development leverage too high to rely on actual emergence |
| 24–60 months | BF | Credibility blend of prior and actual emergence |
| ≥ 60 months | Chain Ladder | Actual development data is credible and dominant |
| Any — new line | Expected Claims | No development history available |